In [0]:
# DBTITLE 1,TEST CELL - Set Parameters Manually (for debugging)
# This cell simulates the parameters that would be passed from LoopConsolidationFiles
# Run this cell first, then run Cell 2 to see exactly where it hangs

# Set widget values manually for testing
dbutils.widgets.text("FileId", "9001", "")
dbutils.widgets.text("CurrentContainer", "/Volumes/pharma_catalog/bronze", "")
dbutils.widgets.text("CurrentFolderPath", "processed_data", "")
dbutils.widgets.text("ConsolidatedLayerDataModel", "MemberDataModel.json", "")
dbutils.widgets.text("ConsolidatedLayerDataModelFilePath", "/Workspace/Users/logi@openhealthagents.org/pharma_bricks/src/etl/datalake-dev/JSON/Consolidation/DataModels", "")
dbutils.widgets.text("ConsolidatedMappingFileName", "ConsolidationMember712.json", "")
dbutils.widgets.text("ConsolidatedMappingFilePath", "/Workspace/Users/logi@openhealthagents.org/pharma_bricks/src/etl/datalake-dev/JSON/Consolidation", "")
dbutils.widgets.text("ConsolidatedFolderPath", "/Volumes/pharma_catalog/bronze/processed_data/consolidated_delta/member", "")

print("✓ Test parameters set. Now run Cell 2 to execute the pipeline logic.")
print("  Watch the output to see exactly where it hangs (if it does).")
print("\nParameters:")
print(f"  FileId: {dbutils.widgets.get('FileId')}")
print(f"  CurrentContainer: {dbutils.widgets.get('CurrentContainer')}")
print(f"  CurrentFolderPath: {dbutils.widgets.get('CurrentFolderPath')}")
print(f"  ConsolidatedFolderPath: {dbutils.widgets.get('ConsolidatedFolderPath')}")

In [0]:
# Databricks notebook source
import json
from pyspark.sql.functions import explode, col, lit, to_date, to_timestamp, expr
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, FloatType, DoubleType, DateType, TimestampType

# Type Mapper mapping definitions
TYPE_MAP = {
    "StringType": StringType(), "IntegerType": IntegerType(), "LongType": LongType(),
    "FloatType": FloatType(), "DoubleType": DoubleType(), "DateType": DateType(), "TimestampType": TimestampType()
}

SQL_TYPE_MAP = {
    "StringType": "STRING", "IntegerType": "INT", "LongType": "BIGINT",
    "FloatType": "FLOAT", "DoubleType": "DOUBLE", "DateType": "DATE", "TimestampType": "TIMESTAMP"
}

def get_sql_expr(row):
    """Dynamic column mapping expression generator with error-tolerant date/timestamp parsing"""
    dt = row["DataType"]
    sql_type = SQL_TYPE_MAP.get(dt, "STRING")
    src = row["SourceColumn"]
    dest = row["DestinationColumn"]
    fmt = row["SourceColumnFormat"]
    q = row["ColumnQuery"]
    
    if q and q.strip():
        return f"NULLIF(CAST({q} AS {sql_type}), '') AS {dest}"
    elif dt == "DateType" and fmt:
        # Use TRY_TO_DATE to return NULL instead of erroring on bad data
        return f"TRY_TO_DATE({src}, '{fmt}') AS {dest}"
    elif dt == "TimestampType" and fmt:
        # Use TRY_TO_TIMESTAMP to return NULL instead of erroring on bad data
        return f"TRY_TO_TIMESTAMP({src}, '{fmt}') AS {dest}"
    else:
        return f"CAST({src} AS {sql_type}) AS {dest}"

In [0]:
def process_consolidation(FileId, CurrentContainer, CurrentFolderPath, 
                         ConsolidatedLayerDataModel, ConsolidatedLayerDataModelFilePath,
                         ConsolidatedMappingFileName, ConsolidatedMappingFilePath,
                         ConsolidatedFolderPath):
    """
    Main consolidation processing function - callable directly to avoid dbutils.notebook.run() concurrency limits.
    Returns JSON string with processing results.
    """
    # Unity Catalog Volume paths - no mount point needed
    FullProcessed = f"{CurrentContainer}/{CurrentFolderPath}/"
    FullConsolidatedFolderPath = f"{ConsolidatedFolderPath}/"
    DataModelFile = f"{ConsolidatedLayerDataModelFilePath}/{ConsolidatedLayerDataModel}"
    ConsolidationMapping = f"{ConsolidatedMappingFilePath}/{ConsolidatedMappingFileName}"

    print(f"Source Path: {FullProcessed}")
    print(f"Output Path: {FullConsolidatedFolderPath}")
    print(f"FileId to process: {FileId}")

    output_response = {}

    try:
        # Safely retrieve Job Context IDs on Serverless
        try:
            ctx = dbutils.notebook.getContext()
            job_id = ctx.tags().get("jobId", "LocalRun")
        except:
            job_id = "LocalRun"
            
        output_response["CurrentJobId"] = job_id

        print("Step 1: Loading data model and mapping files...")
        # 1. Parse Schema and Data Models
        temp_dm = spark.read.format("json").option("multiline", "true").load(DataModelFile)
        data_model = temp_dm.select(explode(col("Fields")).alias("col")).select("col.FieldName", "col.DataType", "col.Ordinal")
        
        # Generate schema structural layout
        fields_list = [StructField(r["FieldName"], TYPE_MAP.get(r["DataType"], StringType()), True) for r in data_model.collect()]
        dest_schema = StructType(fields_list)
        df_data_model = spark.createDataFrame([], dest_schema)
        print(f"  ✓ Data model loaded with {len(fields_list)} fields")

        # 2. Parse Mapping rules
        consolidated_mappings = spark.read.format("json").option("multiline", "true").load(ConsolidationMapping)
        temp_mappings = consolidated_mappings.select(explode(col("columnMapping")).alias("col")).select("col.recordType", "col.selectColumns")
        s_record_type = temp_mappings.select(explode(col("recordType")).alias("col")).select("col.Field", "col.Value")
        s_columns = temp_mappings.select(explode(col("selectColumns")).alias("col")).select("col.SourceColumn", "col.DestinationColumn", "col.SourceColumnFormat", "col.ColumnQuery")

        # Join and generate projection arrays
        seq_columns = data_model.join(s_columns, data_model["FieldName"] == s_columns["DestinationColumn"], "inner")
        select_exprs = [get_sql_expr(r) for r in seq_columns.collect()]
        print(f"  ✓ Mapping rules loaded with {len(select_exprs)} column transformations")

        # Parse Record filters
        filter_rows = s_record_type.collect()
        field = filter_rows[-1]["Field"] if filter_rows and filter_rows[-1]["Field"] else "FileId"
        value = filter_rows[-1]["Value"] if filter_rows and filter_rows[-1]["Value"] else FileId
        print(f"  ✓ Filter: {field} == {value}")

        # 3. Process data
        print(f"\nStep 2: Reading source parquet from {FullProcessed}...")
        df_file_reformatted = spark.read.format("parquet").load(FullProcessed).selectExpr(*select_exprs)
        print("  ✓ Source data read successfully")
        
        print(f"\nStep 3: Filtering for FileID={FileId} and {field}={value}...")
        df_filtered = df_file_reformatted.filter(col("FileID") == FileId).filter(col(field) == value)

        # Re-align columns to map schema destination target definitions
        available_cols = df_filtered.columns
        aligned_exprs = [col(c) if c in available_cols else lit(None).alias(c) for c in df_data_model.columns]
        df_file = df_data_model.union(df_filtered.select(*aligned_exprs))

        # Serverless-compatible data check: use limit + count instead of tail()
        print("\nStep 4: Checking if filtered data has rows...")
        row_count = df_file.count()
        print(f"  ✓ Found {row_count} rows to write")
        
        if row_count > 0:
            print(f"\nStep 5: Writing {row_count} rows to Delta at {FullConsolidatedFolderPath}...")
            df_file.write.format("delta").option("mergeSchema", "true").mode("append").save(FullConsolidatedFolderPath)
            output_response["ConsolidatedCount"] = row_count
            print("  ✓ Write completed successfully")
        else:
            print("  No rows matched the filter criteria - skipping write")
            output_response["ConsolidatedCount"] = 0

        output_response["Status"] = "SUCCESS"
        output_response["ErrorMessage"] = ""
        print("\nPipeline completed successfully!")

    except Exception as e:
        error_msg = str(e)
        print(f"\nERROR: {error_msg}")
        output_response["ConsolidatedCount"] = 0
        output_response["Status"] = "FAILED"
        output_response["ErrorMessage"] = error_msg

    return json.dumps(output_response)

In [0]:
# Support legacy dbutils.notebook.run() calls by reading widget parameters
# This allows the notebook to work both ways (function call or notebook.run)

# Setup Widget Parameters
dbutils.widgets.text("FileId", "", "")
dbutils.widgets.text("CurrentContainer", "", "")
dbutils.widgets.text("CurrentFolderPath", "", "")
dbutils.widgets.text("ConsolidatedLayerDataModel", "", "")
dbutils.widgets.text("ConsolidatedLayerDataModelFilePath", "", "")
dbutils.widgets.text("ConsolidatedMappingFileName", "", "")
dbutils.widgets.text("ConsolidatedMappingFilePath", "", "")
dbutils.widgets.text("ConsolidatedFolderPath", "", "")

FileId = dbutils.widgets.get("FileId")

# Only run if FileId widget is populated (means we were called via dbutils.notebook.run)
if FileId:
    CurrentContainer = dbutils.widgets.get("CurrentContainer") 
    CurrentFolderPath = dbutils.widgets.get("CurrentFolderPath")   
    ConsolidatedLayerDataModel = dbutils.widgets.get("ConsolidatedLayerDataModel") 
    ConsolidatedLayerDataModelFilePath = dbutils.widgets.get("ConsolidatedLayerDataModelFilePath") 
    ConsolidatedMappingFileName = dbutils.widgets.get("ConsolidatedMappingFileName") 
    ConsolidatedMappingFilePath = dbutils.widgets.get("ConsolidatedMappingFilePath")
    ConsolidatedFolderPath = dbutils.widgets.get("ConsolidatedFolderPath")
    
    # Call the function and exit (for backward compatibility)
    result = process_consolidation(
        FileId, CurrentContainer, CurrentFolderPath,
        ConsolidatedLayerDataModel, ConsolidatedLayerDataModelFilePath,
        ConsolidatedMappingFileName, ConsolidatedMappingFilePath,
        ConsolidatedFolderPath
    )
    print(f"\nReturning result: {result}")
    dbutils.notebook.exit(result)
else:
    print("Notebook loaded as module via %run - process_consolidation() function is available for direct calls")